# 0. Install the API tools

In [ ]:
!pip install brainstem_python_api_tools

In [ ]:
from brainstem_api_tools import BrainstemClient

# 1. Client Setup and Authentication

Authentication uses a **browser-based device authorization flow** that supports two-factor authentication (2FA). No credentials are entered into this tool.

On the first run, a browser window opens for you to approve access. The token is then cached at `~/.config/brainstem/token` and reused automatically on subsequent runs.

In [ ]:
# First run: opens a browser window for secure login.
# Subsequent runs reuse the cached token automatically.
client = BrainstemClient()

# 2. Alternative Authentication Options

You can also pass a token directly, use headless mode (prints a URL + code instead of opening a browser), or point the client at a different server.

In [ ]:
# Pass a token directly to skip the browser flow:
# client = BrainstemClient(token="YOUR_TOKEN")

# Headless mode — prints a URL + code to enter manually (useful in remote/CI environments):
# client = BrainstemClient(headless=True)

# Connect to a local development server:
# client = BrainstemClient(url="http://127.0.0.1:8000/")

# 3. Loading Public Data

The BrainSTEM platform hosts both public and private data. Public data can be accessed without authentication. Use `portal="public"` to query public repositories. This is useful for exploring datasets like the Allen Institute's Visual Coding – Neuropixels project.

In [ ]:
# Load public projects (no authentication needed for public portal)
public_projects = client.load("project", portal="public").json()

# Print the number of available public projects
print(f"Found {len(public_projects.get('projects', []))} public projects")

# Display the first few projects
print("\nSample public projects:")
for i, project in enumerate(public_projects.get('projects', [])[:3]):
    print(f"{i+1}. {project.get('name', 'Unnamed')}")
    print(f"   Description: {project.get('description', 'No description')[:100]}...")
    print("")

In [ ]:
# Load your private projects (requires authentication — client must be initialised with a valid token)
# private_projects = client.load("project").json()  # default portal is "private"
# print(f"Found {len(private_projects.get('projects', []))} private projects")

# 4. Filtering

The API supports powerful filtering capabilities. You can filter by exact matches, partial text matches (case-sensitive or insensitive), and various other criteria. Multiple filters on different fields are combined with AND logic.

### Filter Modifiers

- `attribute.contains`: Case-sensitive partial match
- `attribute.icontains`: Case-insensitive partial match
- `attribute.iexact`: Case-insensitive exact match
- `attribute.startswith`: Case-sensitive prefix match
- `attribute.endswith`: Case-sensitive suffix match
- `attribute.istartswith`: Case-insensitive prefix match
- `attribute.iendswith`: Case-insensitive suffix match

In [ ]:
# Case-insensitive name filter
filtered_projects = client.load(
    "project",
    portal="public",
    filters={'name.icontains': 'Allen'}
).json()
print(f"Found {len(filtered_projects.get('projects', []))} projects matching 'Allen'")

# Filters on different fields are combined with AND logic
# Note: a Python dict cannot have two identical keys — use different field names to combine filters
multi_filter = client.load(
    "project",
    portal="public",
    filters={
        'name.icontains': 'Allen',
        'description.icontains': 'Neuropixels'
    }
).json()

print(f"Found {len(multi_filter.get('projects', []))} projects matching both criteria")

In [ ]:
# 5. Retrieving Related Data
# Find the Allen Institute: Visual Coding – Neuropixels project by exact name
allen_projects = client.load(
    "project",
    portal="public",
    filters={'name.iexact': 'Allen Institute: Visual Coding \u2013 Neuropixels'}
).json()

project_id = allen_projects['projects'][0]['id']

# Load the project by ID to get its full details
project_details = client.load("project", portal="public", id=project_id).json()
project = project_details['project']
print(f"Project: {project['name']}")
print(f"Description: {project['description'][:200]}...")

# Load subjects belonging to this project
subjects = client.load(
    "subject",
    portal="public",
    filters={'projects': project_id}
).json()
print(f"\nSubjects: {len(subjects['subjects'])}")

# Load sessions and include experiment data
sessions = client.load(
    "session",
    portal="public",
    filters={'projects': project_id},
    include=['dataacquisition', 'behaviors']
).json()
print(f"Sessions: {len(sessions['sessions'])}")

# Show details for the first session
if sessions['sessions']:
    s = sessions['sessions'][0]
    print(f"\nFirst session: {s['name']}")
    print(f"  Data acquisitions: {len(s.get('dataacquisition', []))}")

# 6. Sorting

You can control the order of returned results using the `sort` parameter. Sort by any field in ascending order or use a `-` prefix for descending order.

In [ ]:
# Sort projects alphabetically by name (ascending)
alpha_projects = client.load("project", portal="public", sort=['name']).json()
print("Alphabetically sorted:")
for p in alpha_projects.get('projects', [])[:3]:
    print(f"  {p['name']}")

# Sort descending
reverse_alpha = client.load("project", portal="public", sort=['-name']).json()
print("\nReverse alphabetical:")
for p in reverse_alpha.get('projects', [])[:3]:
    print(f"  {p['name']}")

# 7. Pagination

By default the API returns one page of results. Use `limit` and `offset` for manual pagination, or `load_all=True` to automatically fetch every page and return a combined dict.

In [ ]:
# Manual pagination: fetch records 1-10, then 11-20
page1 = client.load("project", portal="public", limit=10, offset=0).json()
page2 = client.load("project", portal="public", limit=10, offset=10).json()
print(f"Page 1: {len(page1['projects'])} projects")
print(f"Page 2: {len(page2['projects'])} projects")

# Auto-pagination: fetch ALL records across all pages in one call
all_projects = client.load("project", portal="public", load_all=True)
print(f"\nTotal public projects (all pages): {len(all_projects['projects'])}")

# 8. Convenience Loaders

Convenience loaders are shortcut methods for the most common models. They set sensible default `include` relations and expose named keyword arguments for the most frequent filter fields.

In [ ]:
# load_project — embeds sessions, subjects, collections, cohorts by default
projects = client.load_project(portal="public", name="Allen", load_all=True)
print(f"Projects matching 'Allen': {len(projects['projects'])}")

# load_session — embeds dataacquisition, behaviors, manipulations, epochs by default
# (use portal="public" for public data, or omit for private)
# sessions = client.load_session(name="Rat", load_all=True)

# load_subject — embeds procedures and subjectlogs by default
# subjects = client.load_subject(sex="M", projects="<project-uuid>", load_all=True)

# load_behavior / load_dataacquisition / load_manipulation — scope by session UUID
# behaviors = client.load_behavior(session="<session-uuid>", load_all=True)
# dataacquisition = client.load_dataacquisition(session="<session-uuid>", load_all=True)
# manipulations = client.load_manipulation(session="<session-uuid>", load_all=True)

# load_procedure — scopes by subject UUID
# procedures = client.load_procedure(subject="<subject-uuid>", load_all=True)

# load_collection / load_cohort
# collection = client.load_collection(name="MyCollection")
# cohort = client.load_cohort(name="MyCohort")

# All convenience loaders accept extra filters= and a custom include= override:
# sessions = client.load_session(
#     name="Rat",
#     include=["projects"],
#     filters={"description.icontains": "hippocampus"},
#     load_all=True,
# )

# 9. Creating and Updating Data

Use `client.save()` to create or update records. Omit `id` to create a new record (POST); provide `id` to update an existing one (PATCH). Write operations require authentication on the private portal.

In [ ]:
# Create a new session (replace project UUID with your own)
new_session = client.save(
    "session",
    data={
        "name": "My new session",
        "description": "Created via the Python API",
        "projects": ["<your-project-uuid>"],
    }
).json()
# print(new_session)

# Update an existing session (replace UUID with your own)
updated = client.save(
    "session",
    id="<your-session-uuid>",
    data={"description": "Updated description"}
).json()
# print(updated)

# Create a new subject
new_subject = client.save(
    "subject",
    data={
        "name": "Sub001",
        "sex": "M",
        "projects": ["<your-project-uuid>"],
    }
).json()
# print(new_subject)

print("Replace the placeholder UUIDs above with your own IDs before running.")

# 10. Deleting Data

Use `client.delete()` with a model name and record UUID. A successful deletion returns HTTP 204 (no content). This operation is **permanent** — use with care.

In [ ]:
# Delete a record by UUID (replace with your own ID)
response = client.delete("session", id="<your-session-uuid>")

if response.status_code == 204:
    print("Session deleted successfully.")
else:
    print(f"Delete failed: {response.status_code} — {response.text}")

print("Replace the placeholder UUID above with your own ID before running.")